# Langchain Agents 
This notebook demonstrates how to create and use agents in Langchain, including the use of tools like DuckDuckGoSearch for real-time information retrieval.

At the end of this notebook, we can see how to create a ReAct agent from a prompt template and use it with a memory system to maintain chat history.



In [2]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq


In [3]:
# load environment variables from .env file
load_dotenv()

True

In [4]:
# Get llms models via the `langchain_groq` package
llm_model = ChatGroq(
    model="llama-3.3-70b-versatile", #llama-3.3-70b-versatile",
    temperature=.5,
    max_tokens=512,
)

## Langchain Tools

The [third-party tools](https://python.langchain.com/docs/integrations/tools/) from the langchain-community ecosytem allow us to extent the capabilities of our agents

In [5]:
# from langchain_community.tools import DuckDuckGoSearchRun
from ddgs import DDGS
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

`DuckDuckGoSearchRun` allows us to perform web searches and retrieve information in real-time, which can be very useful for agents that need to answer questions based on current data.


In [6]:
#api_wrapper = DuckDuckGoSearchAPIWrapper(region="en-en",max_results=10)
#search = DuckDuckGoSearchRun()#,api_wrapper=api_wrapper)

def ddg_search(query: str):
    with DDGS() as ddgs:
        return ddgs.text(query, max_results=5)
search = ddg_search

In [7]:
results = search("Prime Italian Minister")  

In [8]:
results

[{'title': 'Prime Minister of Italy - Wikipedia',
  'href': 'https://en.wikipedia.org/wiki/Prime_Minister_of_Italy',
  'body': 'The Italian prime minister presided over a very unstable political system as in its first sixty years of existence (1861–1921) Italy changed its ...'},
 {'title': 'List of prime ministers of Italy - Wikipedia',
  'href': 'https://en.wikipedia.org/wiki/List_of_Prime_Ministers_of_Italy',
  'body': 'Top right: Benito Mussolini was the longest-serving prime minister in Italian history. ... was the longest-serving prime minister of the Italian ...'},
 {'title': 'Italian Prime Minister',
  'href': 'https://wn.com/Italian_Prime_Minister',
  'body': 'Italian Prime Minister Giorgia Meloni was busted giving an epic eye roll at the G7 summit — setting social media alight with her less-than ...'},
 {'title': 'Giorgia Meloni sworn in as Italian PM – but questions over',
  'href': 'https://www.telegraph.co.uk/world-news/2022/10/22/giorgia-meloni-sworn-italian-pm-questions-l

In [9]:
# Define a tool for the DuckDuckGo search
from langchain.tools import Tool
search_tool = Tool(
    name="DuckDuckGoSearch",
    func=ddg_search,
    description="Use this tool when you need to search for real-time information from DuckDuckGo."
)

In [10]:
from langchain.agents import initialize_agent, AgentType


In [11]:
# Initialize the agent with the DuckDuckGo search tool
agent = initialize_agent(
    tools=[search_tool],
    llm=llm_model,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    max_iterations=7,
    agent_name="DuckDuckGoSearchAgent",
    agent_description="An agent that can search the web using DuckDuckGoSearch and summarize findings clearly."
)

# Now we can use the agent to answer questions
results = agent.invoke("Who is the Italian Prime Minister?")  # Example query to the agent

/opt/anaconda3/envs/ds/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 0.3.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  warn_deprecated(




> Entering new AgentExecutor chain...
To find out who the current Italian Prime Minister is, I should search for the most recent information available. 

Action: DuckDuckGoSearch
Action Input: Italian Prime Minister
Observation: [{'title': 'Italian Prime Minister', 'href': 'https://en.wikipedia.org/wiki/Italian_Prime_Minister', 'body': "The prime minister of Italy, officially the president of the Council of Ministers (Italian: presidente del Consiglio dei ministri), is the head of government of the Italian Republic. The office of president of the Council of Ministers is established by articles 92–96 of the Constitution of Italy; the president of the Council of Ministers is appointed by the president of the Italian Republic and must have the confidence of the parliament to stay in office.Prior to the establishment of the Italian Republic, the position was known as the president of the Council of Ministers of the Kingdom of Italy (Presidente del Consiglio dei ministri del Regno d'Itali

### Create a an Agent with DuckDuckGoSearch with Memory
In this section, we will create an agent that can search the web using DuckDuckGoSearch and maintain a memory of previous interactions. 

In [12]:
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder


In [13]:

# Build a chat prompt that includes conversation history so the agent can resolve references.
prompt_template = ChatPromptTemplate.from_messages([
    ("system", """Answer the following questions as best you can. You have access to the following tools:
{tools}
Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    ("assistant", "{agent_scratchpad}"),
])


The react_template instructs the laguage model to:
1. __Think__ about how to solve a user's question
2. __Use tools__ (actions) to gather mor information or perform actions
3. __Observe results__ and update the its thoughts
4. Eventually, reach a __Final Answer__

The following is the explanation of the `react_template` used to create the agent:

```python
react_template = """
'Answer the following questions as best you can. You have access to the following tools:

{tools}
```
+ Introduce the agent and tells it what tools it can use
+ {tools} is a placeholder for the tools that will be used by the agent

```text
Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question
```

This is the strict __format__ the LLM must follow to:

* Show its reasoning (Thought)

* Choose and execute a tool (Action)

* Feed it input (Action Input)

* Observe results (Observation)

* Loop until it gets to the final answer

```text
Begin!
Question: {input}
Thought:{agent_scratchpad}'
"""
```
+ `{input}` is the actual user question.

+ `{agent_scratchpad}` holds any intermediate steps already completed (e.g., previous Thoughts, Actions, Observations).

+ The agent continues from this point.

In [14]:
# prompt_template defined above with chat history support


In [15]:
from langchain.agents import AgentExecutor, create_react_agent
from langchain.memory import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [16]:
# Simple session store to keep track of chat history per conversation
store = {}


def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


In [17]:
# Create the agent 
agent = create_react_agent(llm_model, [search_tool], prompt_template)
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool],
    verbose=True,
    handle_parsing_errors=True,
)


In [18]:
# Wrap the agent executor with message history
agent_with_chat_history = RunnableWithMessageHistory(
    agent_executor,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)


In [19]:
# Now we can use the agent with chat history to answer questions
agent_with_chat_history.invoke(
    {"input": "How many people live in italy?"},
    config={"configurable": {"session_id": "1"}},
)

Parent run 7d5c9fbc-2714-45bc-ba45-ac26b4de3eaa not found for run 81be7a90-6744-4868-9125-fa375c24b0e7. Treating as a root run.




> Entering new AgentExecutor chain...
Thought: To find the current population of Italy, I should search for the most recent data available.

Action: DuckDuckGoSearch
Action Input: Italy population 2024
[{'title': 'Demographics of Italy - Wikipedia', 'href': 'https://en.wikipedia.org/wiki/Demographics_of_Italy', 'body': 'Animated population pyramid 1982-2021. Those born during the World wars are marked in dark The demography of Italy is monitored by the Italian National Institute of Statistics (Istat). At the beginning of 2024, Italy had an estimated population of 58.9 million. [1] Its population density, at 195.7 inhabitants per square kilometre (507/sq mi), is higher than both the EU (106.6/km 2) and ...'}, {'title': 'Italy Population (2026) - Worldometer', 'href': 'https://www.worldometers.info/world-population/italy-population/', 'body': 'Population of Italy: current, historical, and projected population, growth rate, immigration, median age, total fertility rate (TFR), population

{'input': 'How many people live in italy?',
 'chat_history': [],
 'output': 'The estimated population of Italy at the beginning of 2024 is approximately 58.9 million.'}

In [20]:
# Let's ask another question to see how the agent uses the chat history
agent_with_chat_history.invoke(
    {"input": "what is their national anthem called?"},
    config={"configurable": {"session_id": "1"}},
)

Parent run 4c734308-6260-4a02-8fa2-dffe78c911e1 not found for run a0958f63-593d-4e9f-9abe-927dfc633055. Treating as a root run.




> Entering new AgentExecutor chain...
Question: what is the national anthem of Italy called?
Thought: I should search for the national anthem of Italy to find its name.
Action: DuckDuckGoSearch
Action Input: Italy national anthem[{'title': 'Italy national anthem', 'href': 'https://en.wikipedia.org/wiki/Italy_national_anthem', 'body': '"Il Canto degli Italiani" is a patriotic song written by Goffredo Mameli and set to music by Michele Novaro in 1847, currently used as the national anthem of Italy. It is known among Italians as the "Inno di Mameli"—after the author of the lyrics—or as "Fratelli d\'Italia"—from the song\'s opening line. The piece, in 44 time signature and B-flat major key, has six strophes, and a refrain sung after each. The sixth group of verses, almost never performed, recalls the text of the first strophe. IWSC number T-200.797.021-7 (Fraltelli d\'Italia)The song was popular during the unification of Italy and the following decades. However, after the proclamation of

{'input': 'what is their national anthem called?',
 'chat_history': [HumanMessage(content='How many people live in italy?'),
  AIMessage(content='The estimated population of Italy at the beginning of 2024 is approximately 58.9 million.')],
 'output': 'The national anthem of Italy is called "Il Canto degli Italiani".'}

### Exercise
Create or Select some tools from the [langchain-community tools](https://docs.langchain.com/oss/python/integrations/tools) and create a ReAct agent that can use them to solve certain tasks.

In [38]:
from ddgs import DDGS

def ddg_search(query: str) -> str:
    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=5):
            results.append(r["body"])
    return "\n".join(results)

In [39]:
from langchain.tools import Tool

search_tool = Tool(
    name="WebSearch",
    func=ddg_search,
    description="Use this tool to search the internet for current or factual information"
)

In [40]:
def extract_facts_llm(text: str) -> str:
    return llm_model.invoke(
        f"Extract 3-5 key factual bullet points from this text:\n{text}"
    )

In [41]:
fact_tool = Tool(
    name="FactExtractor",
    func=extract_facts_llm,
    description="Use this tool to extract key facts from a given text"
)

In [42]:
import re

def calculate(expr: str) -> str:
    expr = expr.replace(",", "")
    expr = re.sub(r"\(.*?\)", "", expr)  # убираем комментарии
    expr = expr.lower()
    expr = expr.replace("million", "*1_000_000")
    expr = expr.replace("billion", "*1_000_000_000")

    try:
        return str(eval(expr))
    except Exception as e:
        return f"Error: {e}"

math_tool = Tool(
    name="Calculator",
    func=calculate,
    description="Use this tool for math calculations"
)

In [43]:
from langchain.agents import initialize_agent, AgentType

agent = initialize_agent(
    tools=[search_tool, fact_tool, math_tool],
    llm=llm_model,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    max_iterations=5
)

In [44]:
agent.invoke("Who is the current president of France and what are key facts about them?")



> Entering new AgentExecutor chain...
To find the current president of France and key facts about them, I should start by searching the internet for the most up-to-date information.

Action: WebSearch
Action Input: current president of France
Observation: 1 week ago - Emmanuel Jean-Michel Frédéric Macron (born 21 December 1977) is a French politician who has served as President of France and Co-Prince of Andorra since 2017. He served as Minister of Economics and Finance under President François Hollande from 2014 to 2016.
2 weeks ago - The officeholder is also honorary proto-canon of the Archbasilica of Saint John Lateran in Rome, although some have rejected the title in the past. The current president is Emmanuel Macron, who succeeded François Hollande on 14 May 2017 following the 2017 presidential election, and was inaugurated ...
1 week ago - A 1962 referendum held under the ... vote. Since then, ten presidential elections have taken place. The 25th and current officeholder has be

{'input': 'Who is the current president of France and what are key facts about them?',
 'output': "The current president of France is Emmanuel Macron, born on December 21, 1977. He has served as President of France and Co-Prince of Andorra since 2017 and was previously the Minister of Economics and Finance under President François Hollande from 2014 to 2016. He was re-elected in 2022 and has faced international challenges during his second term, including the Russian invasion of Ukraine and efforts to strengthen Europe's security posture."}

In [45]:
agent.invoke("What is the population of Germany and how does it compare to France?")



> Entering new AgentExecutor chain...
To find the population of Germany and compare it to France, I need to first find the current population of both countries. 

Action: WebSearch
Action Input: population of Germany and France
Observation: This list of European countries by population comprises the 51 countries and 5 territories and dependencies in Europe, broadly defined, including Cyprus, Kazakhstan, Turkey, and the countries of the Caucasus. The most populous European country is Russia, with a population of over 110 million in its European part (144 million overall). Turkey, with a population of about 85 million, has most of ...
List of countries (or dependencies) in Europe ranked by population, from the most populated. Growth rate, median age, fertility rate, area, density, population density, urbanization, urban population, share of world population.
The latest officially reported population of France was 68,551,653 in 2024 vs 83,516,593 people in Germany in 2024. In 2026, base

{'input': 'What is the population of Germany and how does it compare to France?',
 'output': 'The population of Germany is approximately 82,619,736, and the population of France is approximately 68,756,508. The difference in population between the two countries is around 13,863,228. Germany has a larger population than France.'}

In [46]:
agent.invoke("Tell me about Tesla company and extract key facts")



> Entering new AgentExecutor chain...
To provide information about Tesla and extract key facts, I should first search for an overview of the company on the internet.

Action: WebSearch
Action Input: Tesla company overview
Observation: Tesla Company is a multinational company based in Austin city, Texas State, in the country of USA. ... The following year Tesla Company was began ...
With experience at companies like Tesla, AGL Energy, and National Australia Bank, Cleve has led various teams and initiatives to support sustainable ...
In 2012, Tesla company organised Superchargers in the USA, and Europe created for charging batteries fast and at no other value to Tesla buyers.
Originally known as Tesla Motors, the company rebranded to Tesla in 2017 to reflect its expanded focus beyond automotive manufacturing.
Tesla is an automotive company focusing on electric vehicles, clean energy solutions, and advanced automotive technology.
Thought:To provide more detailed information about Tesla 

{'input': 'Tell me about Tesla company and extract key facts',
 'output': "Tesla is an automotive company focusing on electric vehicles, clean energy solutions, and advanced automotive technology. The key facts about the company are: \n* Tesla is an automotive company that focuses on electric vehicles, clean energy solutions, and advanced automotive technology.\n* The company was originally known as Tesla Motors and rebranded to Tesla in 2017.\n* In 2012, Tesla introduced Superchargers in the USA and Europe to provide fast and free charging for its customers.\n* Tesla's Superchargers are available for Tesla buyers, implying that the service is exclusive to the company's customers."}